# Blender on Colab (Enhanced Headless Cloud Rendering)
Features:
- **Colab Forms Interface**: A clean GUI to set up your render without typing inputs.
- **Auto-downloads latest Blender** if none exists in your Google Drive.
- **Installs Addons automatically** (Place `.zip` files or folders in `Colab-Render/addons/`).
- **OptiX & Multi-GPU Support** for 2x faster Cycles rendering.
- **Auto-Compositing**: Automatically saves your Composite, Base Image, and Noisy Image into separate files simultaneously.
- **Auto-Resume**: If your Colab crashes mid-render, simply run it again and it will resume from where it left off!

In [ ]:
# Step-1: Connect Google Drive to Colab
from google.colab import drive
import os
import glob
import shutil
import zipfile
import subprocess
import urllib.request
import re

drive.mount('/content/drive')

gdrive_path = "/content/drive/MyDrive/Colab-Render"
output_path = f"{gdrive_path}/Image Sequence"
addons_dir = f"{gdrive_path}/addons"

# Ensure folders exist
os.makedirs(output_path, exist_ok=True)
os.makedirs(addons_dir, exist_ok=True)

In [ ]:
# Step-2: Find or Download Blender
blender_archives = glob.glob(f"{gdrive_path}/blender-*.tar.xz")

if blender_archives:
    blender_archive = sorted(blender_archives)[-1]
    print(f"Found local Blender archive: {blender_archive}")
else:
    print("No Blender archive found in Google Drive. Fetching the latest version from official servers...")
    url = "https://download.blender.org/release/"
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    html = urllib.request.urlopen(req).read().decode('utf-8')
    versions = re.findall(r'href="Blender(\d+\.\d+)/"', html)
    versions.sort(key=lambda s: list(map(float, s.split('.'))))
    latest_version = versions[-1]
    
    url2 = f"{url}Blender{latest_version}/"
    req2 = urllib.request.Request(url2, headers={'User-Agent': 'Mozilla/5.0'})
    html2 = urllib.request.urlopen(req2).read().decode('utf-8')
    files = re.findall(r'href="(blender-.*?-linux-x64\.tar\.xz)"', html2)
    latest_file = sorted(files)[-1]
    download_url = f"{url2}{latest_file}"
    blender_archive = f"/content/{latest_file}"
    
    print(f"Downloading {latest_file}... this will take a moment.")
    !wget -q --show-progress {download_url} -O {blender_archive}
    print("Download complete.")

blender_folder_name = os.path.basename(blender_archive).replace('.tar.xz', '')
print("Extracting to /content... this might take a minute.")

!tar xf "{blender_archive}" -C "/content"
blender_path = f"/content/{blender_folder_name}/blender"
print(f"Blender extracted at: {blender_path}")

In [ ]:
# Step-3: Install Addons
blender_addons_dest = glob.glob(f"/content/{blender_folder_name}/*/scripts/addons")

if blender_addons_dest:
    blender_addons_dest = blender_addons_dest[0]
    detected_modules = []
    for item in os.listdir(addons_dir):
        item_path = os.path.join(addons_dir, item)
        if item.endswith(".zip"):
            print(f"Extracting addon zip: {item}")
            with zipfile.ZipFile(item_path, 'r') as zip_ref:
                top_levels = set([name.split('/')[0] for name in zip_ref.namelist()])
                for tl in top_levels:
                    if tl.endswith(".py"):
                        detected_modules.append(tl[:-3])
                    elif tl:
                        detected_modules.append(tl)
                zip_ref.extractall(blender_addons_dest)
        elif os.path.isdir(item_path):
            dest = os.path.join(blender_addons_dest, item)
            detected_modules.append(item)
            if not os.path.exists(dest):
                shutil.copytree(item_path, dest)
                print(f"Copied addon folder: {item}")

    if detected_modules:
        print("\n--- Detected Addon Module Names ---")
        print("Copy and paste these into the 'addons_to_enable' field in the next cell (comma-separated):")
        for mod in set(detected_modules):
            print(f"- {mod}")
else:
    print("Warning: Could not find Blender addons directory.")

In [ ]:
# @title ⚙️ Configure & Render { display-mode: "form" }
blend_file_name = "Untitled.blend" # @param {type:"string"}
render_engine = "CYCLES" # @param ["CYCLES", "EEVEE", "WORKBENCH"]
render_type = "animation" # @param ["animation", "image"]
addons_to_enable = "" # @param {type:"string"}

# @markdown ---
# @markdown ### Frame Settings
image_frame = 1 # @param {type:"integer"}
start_frame = 1 # @param {type:"integer"}
end_frame = 250 # @param {type:"integer"}

# Resolve paths
blend_file = f"{gdrive_path}/{blend_file_name}"
engine_map = {"CYCLES": "CYCLES", "EEVEE": "BLENDER_EEVEE_NEXT", "WORKBENCH": "BLENDER_WORKBENCH"}
engine_prefix = "Cycles" if render_engine == "CYCLES" else "Eevee" if render_engine == "EEVEE" else "Workbench"
output_file_pattern = f"{output_path}/{engine_prefix}_####"

# Generate setup script
setup_script = f"""
import bpy
import os

# Enable Addons
addons = '{addons_to_enable}'.split(',')
for addon in addons:
    addon = addon.strip()
    if addon:
        try:
            bpy.ops.preferences.addon_enable(module=addon)
            print(f"Successfully enabled addon: {{addon}}")
        except Exception as e:
            print(f"Failed to enable addon {{addon}}: {{e}}")

# Set Render Engine
bpy.context.scene.render.engine = '{engine_map.get(render_engine, "CYCLES")}'
print(f"Render engine set to {engine_map.get(render_engine, "CYCLES")}")

# Auto-Compositing Setup
try:
    bpy.context.scene.use_nodes = True
except:
    pass
if hasattr(bpy.context.scene, 'node_tree'):
    tree = bpy.context.scene.node_tree
else:
    if not bpy.context.scene.compositing_node_group:
        bpy.context.scene.compositing_node_group = bpy.data.node_groups.new(name='CompositorTree', type='CompositorNodeTree')
    tree = bpy.context.scene.compositing_node_group
rl = next((node for node in tree.nodes if node.type == 'R_LAYERS'), None)
if not rl:
    rl = tree.nodes.new(type="CompositorNodeRLayers")

comp_node = next((node for node in tree.nodes if node.type in ['COMPOSITE', 'GROUP_OUTPUT']), None)
if not comp_node:
    try:
        comp_node = tree.nodes.new(type="CompositorNodeComposite")
    except:
        comp_node = tree.nodes.new(type="NodeGroupOutput")
        if hasattr(tree, 'interface'):
            existing = [s for s in tree.interface.items_tree if getattr(s, 'in_out', None) == 'OUTPUT' and s.name == 'Image']
            if not existing:
                tree.interface.new_socket(name='Image', socket_type='NodeSocketColor', in_out='OUTPUT')
    try:
        if 'Image' in rl.outputs:
            tree.links.new(rl.outputs['Image'], comp_node.inputs.get('Image', comp_node.inputs[0]))
    except:
        pass

if bpy.context.scene.render.engine == 'CYCLES':
    try:
        bpy.context.scene.view_layers[0].use_pass_noisy_image = True
    except:
        try:
            bpy.context.scene.view_layers[0].use_denoising_data = True
        except:
            pass
    
    file_output = tree.nodes.new(type="CompositorNodeOutputFile")
    if hasattr(file_output, 'base_path'):
        file_output.base_path = f"{output_path}/"
        try:
            file_output.format.file_format = 'PNG'
        except TypeError:
            pass
        file_output.file_slots.clear()
        file_output.file_slots.new("Composite_####")
        file_output.file_slots.new("Noisy_####")
        file_output.file_slots.new("Base_####")
    else:
        file_output.directory = f"{output_path}/"
        try:
            file_output.format.file_format = 'PNG'
        except TypeError:
            pass
        file_output.file_output_items.clear()
        file_output.file_output_items.new('RGBA', "Composite_####")
        file_output.file_output_items.new('RGBA', "Noisy_####")
        file_output.file_output_items.new('RGBA', "Base_####")
    
    links = tree.links
    for out in rl.outputs:
        if out.name == 'Image':
            links.new(out, file_output.inputs["Base_####"])
        elif out.name == 'Noisy Image':
            links.new(out, file_output.inputs["Noisy_####"])
            
    try:
        if comp_node.inputs.get('Image') and comp_node.inputs['Image'].is_linked:
            linked_socket = comp_node.inputs['Image'].links[0].from_socket
            links.new(linked_socket, file_output.inputs["Composite_####"])
        elif comp_node.inputs[0].is_linked:
            linked_socket = comp_node.inputs[0].links[0].from_socket
            links.new(linked_socket, file_output.inputs["Composite_####"])
    except:
        pass

# Enable Multi-GPU & OptiX
if bpy.context.scene.render.engine == 'CYCLES':
    preferences = bpy.context.preferences
    cycles_preferences = preferences.addons['cycles'].preferences
    
    device_type = "OPTIX"
    try:
        cycles_preferences.compute_device_type = "OPTIX"
        cycles_preferences.get_devices()
        has_optix = any(d.type == "OPTIX" for d in cycles_preferences.devices)
        if not has_optix:
            device_type = "CUDA"
    except:
        device_type = "CUDA"
        
    cycles_preferences.compute_device_type = device_type
    print(f'Compute device type set to: {{device_type}}')
    
    for device in cycles_preferences.devices:
        if device.type == device_type:
            device.use = True
            print(f'Enabled device: {{device.name}}')
        else:
            device.use = False
            
    bpy.context.scene.cycles.device = 'GPU'
"""

setup_script_path = f"{gdrive_path}/setup_blender.py"
with open(setup_script_path, "w") as f:
    f.write(setup_script)
print("Generated Blender setup script.")

# Rendering
os.environ["MESA_GL_VERSION_OVERRIDE"] = "4.5"
os.environ["MESA_GLSL_VERSION_OVERRIDE"] = "450"
os.environ["EGL_PLATFORM"] = "x11"
os.environ["DISPLAY"] = ":0"

base_cmd = [
    blender_path, "-b", blend_file,
    "-P", setup_script_path,
    "-noaudio", "-F", "PNG",
    "-o", output_file_pattern
]

cmd = None
if render_type == "image":
    cmd = base_cmd + ["-f", str(image_frame)]
elif render_type == "animation":
    # Auto Resume Logic
    existing_files = glob.glob(f"{output_path}/*.png")
    if existing_files:
        frames = []
        for f in existing_files:
            matches = re.findall(r'\d+', os.path.basename(f))
            if matches:
                frames.append(int(matches[-1]))
        if frames:
            highest_frame = max(frames)
            if start_frame <= highest_frame < end_frame:
                print(f"\nAuto-Resume: Found existing frames up to {highest_frame}. Resuming from {highest_frame + 1}...")
                start_frame = highest_frame + 1
            elif highest_frame >= end_frame:
                print(f"\nAll frames up to {end_frame} are already rendered! Nothing to do.")
                cmd = "SKIP"
    
    if cmd != "SKIP":
        cmd = base_cmd + ["-s", str(start_frame), "-e", str(end_frame), "-a"]
    else:
        cmd = None

if cmd:
    print(f"\nStarting render...\n{' '.join(cmd)}\n")
    import sys
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()
    process.wait()
    if process.returncode == 0:
        print("\n✅ Rendering complete! Frames saved in Google Drive.")
    else:
        print("\n❌ Rendering failed.")